# ANTEPRIMA RAPIDA del report (senza fit, ~1-2 min)

Per mostrare **subito** al team il formato del `report.pdf`, **senza** aspettare il fit Meridian (30-40 min su GPU).

Usa dati sintetici + un **fit di esempio gia' salvato**, e mostra le pagine del report qui sotto.

**Non serve la GPU** (Runtime CPU va bene).

## 1. Setup veloce (niente Meridian/TensorFlow)

In [ ]:
%cd /content
!rm -rf Tesi-MMM
!git clone -q https://github.com/Giacomod2001/Tesi-MMM.git
%cd Tesi-MMM
!pip install -q pdfplumber openpyxl

## 2. Genera i dati sintetici (~1 min, su CPU)

In [ ]:
!python -m pipeline.generator.run > /dev/null 2>&1
from pipeline.ingestion import build as B
from pipeline import config
pr, tb = B.propose_plan(config.RAW_DIR)
for p in pr:
    p.confirmed = True
B.ingest(config.RAW_DIR, plan=pr, interactive=False, tables=tb)
print('Dati pronti.')

## 3. Anteprima del report (pagine inline)

In [ ]:
import os, json
import pandas as pd
from pipeline import config
from pipeline.allocator import run as RUN, report_pdf as R, quarter as Q, campaigns as C2

summary = json.load(open('examples/model_fit_demo.json'))
media = pd.read_csv(os.path.join(config.CANON_DIR, 'media.csv'), parse_dates=['week'])
outcome = pd.read_csv(os.path.join(config.CANON_DIR, 'outcome.csv'))
seas = pd.read_csv(os.path.join(config.CANON_DIR, 'seasonality.csv'), parse_dates=['week'])
nw = media['week'].nunique()
hist = (media.groupby('channel')['spend'].sum() / nw).to_dict()
rev = float(outcome['revenue'].sum() / max(outcome['conversions'].sum(), 1))
alloc = Q.optimize_from_summary(summary, hist, Q.Constraints(total_budget=420000, max_spend={'google': 120000}))
canali = RUN._readable_canali(alloc, summary, hist, rev)
roas = C2.campaign_roas(media, rev)
roi = {c: e['roi']['q50'] for c, e in summary['channels'].items()}
camp = C2.allocate_campaigns(dict(zip(alloc['channel'], alloc['budget_quarter'])), roas, roi, max_shift=0.5)
campagne = RUN._readable_campagne(camp, canali)

R.build_pdf(canali, campagne, summary, media, seas, rev, os.path.join(config.OUTPUT_DIR, 'report.pdf'))
pngs = R.build_pngs(canali, campagne, summary, media, seas, os.path.join(config.OUTPUT_DIR, 'preview'))
from IPython.display import Image, display
for p in pngs:
    display(Image(filename=p))

## 4. (opzionale) Scarica il PDF

In [ ]:
from google.colab import files
files.download('pipeline/data/output/report.pdf')